In [ ]:
import yaml
import os
import polars as pl
from datetime import datetime ,date

with open("../config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

data_dir = config["data_dir"]
print(data_dir  )
if os.path.exists(data_dir) and os.path.isdir(data_dir):
    list_of_files = os.listdir(data_dir)
else:
    list_of_files = []
print(list_of_files)

In [ ]:
# list_of_files = [f for f in list_of_files if f.endswith(".parquet")]
list_of_tickers = [f for f in list_of_files if not f.startswith("_catalog") and f.endswith(".parquet")]
catalog = [f for f in list_of_files if f.startswith("_catalog") and f.endswith(".parquet")][0]
# parquet_path = os.path.join(data_dir, f"{ticker}.parquet")
catalog_path = os.path.join(data_dir, f"{catalog}")

df_catalog = pl.read_parquet(catalog_path)['ticker','end'].sort('ticker')
df_catalog = df_catalog.with_columns(
    pl.col('end').dt.date().alias('date')
)
df_catalog


In [ ]:
df_backtest_runs = (
    df_catalog
    .select("ticker", "date")
    .unique()
    .sort(["ticker", "date"])
)
df_backtest_runs


In [ ]:
tickers_days = {
    row["ticker"]: row["days"]
    for row in (
        df_catalog
        .select(
            "ticker",
            pl.col("end").dt.strftime("%Y-%m-%d").alias("date"),
        )
        .group_by("ticker")
        .agg(
            pl.col("date")
            .unique()
            .sort(descending=True)
            .alias("days")
        )
        .sort("ticker")
        .to_dicts()
    )
}

tickers_days

In [ ]:

columns = ["datetime", "ticker", "open", "high", "low", "close", "volume"]
frames = []

for ticker, days in tickers_days.items():
    parquet_path = os.path.join(data_dir, f"{ticker}.parquet")

    if not os.path.isfile(parquet_path):
        raise FileNotFoundError(f"No existe el paquete Parquet para {ticker}: {parquet_path}")

    requested_days = [date.fromisoformat(day) for day in days]

    ticker_df = (
        pl.read_parquet(parquet_path)
        .filter(pl.col("datetime").dt.date().is_in(requested_days))
        .with_columns(pl.lit(ticker).alias("ticker"))
        .select(columns)
    )

    frames.append(ticker_df)

df = pl.concat(frames).sort(["ticker", "datetime"])
df

In [ ]:
GROUPS = ["ticker", "trading_day"]


def daily_vwap():
    return [
        (
            (pl.col("price") * pl.col("volume")).cum_sum().over(GROUPS)
            / pl.col("volume").cum_sum().over(GROUPS)
        ).alias("vwap_daily")
    ]


def opening_range_15m():
    opening_window = (
        (pl.col("datetime").dt.hour() == 9)
        & (pl.col("datetime").dt.minute() >= 30)
        & (pl.col("datetime").dt.minute() < 45)
    )
    range_ready = (
        (pl.col("datetime").dt.hour() > 9)
        | (
            (pl.col("datetime").dt.hour() == 9)
            & (pl.col("datetime").dt.minute() >= 45)
        )
    )

    range_high = pl.when(opening_window).then(pl.col("high")).max().over(GROUPS)
    range_low = pl.when(opening_window).then(pl.col("low")).min().over(GROUPS)

    return [
        pl.when(range_ready).then(range_high).otherwise(None).alias("orb_high_15m"),
        pl.when(range_ready).then(range_low).otherwise(None).alias("orb_low_15m"),
    ]


def median_vol(period=100):
    median_volume = (
        pl.col("volume").rolling_median(period, min_samples=period).over(GROUPS)
    )
    return [median_volume.alias(f"median_vol_{period}")]


def atr(period=14):
    previous_close = pl.col("close").shift(1).over(GROUPS)

    true_range = pl.max_horizontal(
        pl.col("high") - pl.col("low"),
        (pl.col("high") - previous_close).abs(),
        (pl.col("low") - previous_close).abs(),
    )

    return [
        true_range
        .rolling_mean(window_size=period, min_samples=period)
        .over(GROUPS)
        .alias(f"atr_{period}")
    ]


def bollinger_bands(period=20, std_dev=2.0):
    middle = pl.col("close").rolling_mean(period, min_samples=period).over(GROUPS)
    deviation = (
        pl.col("close")
        .rolling_std(period, min_samples=period, ddof=0)
        .over(GROUPS)
    )

    return [
        middle.alias(f"bb_middle_{period}"),
        (middle + deviation * std_dev).alias(f"bb_upper_{period}"),
        (middle - deviation * std_dev).alias(f"bb_lower_{period}"),
    ]


def rsi(period=14):
    change = pl.col("close").diff().over(GROUPS)
    gain = pl.when(change > 0).then(change).otherwise(0.0)
    loss = pl.when(change < 0).then(-change).otherwise(0.0)

    average_gain = (
        gain.ewm_mean(alpha=1 / period, adjust=False, min_samples=period)
        .over(GROUPS)
    )
    average_loss = (
        loss.ewm_mean(alpha=1 / period, adjust=False, min_samples=period)
        .over(GROUPS)
    )

    return [
        pl.when(average_loss == 0)
        .then(100.0)
        .when(average_gain == 0)
        .then(0.0)
        .otherwise(100.0 - (100.0 / (1.0 + average_gain / average_loss)))
        .alias(f"rsi_{period}")
    ]


def donchian_channels(period=20):
    upper = pl.col("high").rolling_max(period, min_samples=period).over(GROUPS)
    lower = pl.col("low").rolling_min(period, min_samples=period).over(GROUPS)

    return [
        upper.alias(f"donchian_upper_{period}"),
        lower.alias(f"donchian_lower_{period}"),
        ((upper + lower) / 2).alias(f"donchian_middle_{period}"),
    ]


def momentum(period=10):
    return [
        (
            pl.col("close")
            - pl.col("close").shift(period).over(GROUPS)
        ).alias(f"momentum_{period}")
    ]


def add_indicators(frame, indicator_functions):
    expressions = [
        expression
        for indicator_function in indicator_functions
        for expression in indicator_function()
    ]
    return frame.with_columns(expressions)


df_to_plot = (
    df
    .with_columns(
        pl.col("datetime").dt.convert_time_zone("America/New_York").alias("datetime")
    )
    .sort(["ticker", "datetime"])
    .with_columns(
        pl.col("datetime").dt.date().alias("trading_day"),
        ((pl.col("high") + pl.col("low") + pl.col("close")) / 3).alias("price"),
    )
)

df_to_plot = add_indicators(
    df_to_plot,
    [
        daily_vwap,
        opening_range_15m,
        atr,
        bollinger_bands,
        rsi,
        donchian_channels,
        momentum,
        median_vol,
    ],
)

df_to_plot

In [ ]:
# import plotly.graph_objects as go
# from plotly.subplots import make_subplots


# GROUPS = ["ticker", "trading_day"]

# plot_df = (
#     df_to_plot
#     .sort(["ticker", "datetime"])
#     .with_row_index("plot_x")
# )


# def add_segmented_line(fig, frame, column, name, row, color, dash="solid", width=1):
#     """Añade una línea sin conectar distintos tickers ni días."""
#     show_legend = True

#     for segment in frame.partition_by(GROUPS, maintain_order=True):
#         valid_segment = segment.drop_nulls(column)

#         if valid_segment.height == 0:
#             continue

#         fig.add_trace(
#             go.Scatter(
#                 x=valid_segment["plot_x"].to_list(),
#                 y=valid_segment[column].to_list(),
#                 mode="lines",
#                 name=name,
#                 legendgroup=name,
#                 showlegend=show_legend,
#                 line={"color": color, "dash": dash, "width": width},
#             ),
#             row=row,
#             col=1,
#         )

#         show_legend = False


# volume_colors = [
#     "red" if close > open_price
#     else "green" if close < open_price
#     else "gray"
#     for open_price, close in zip(
#         plot_df["open"].to_list(),
#         plot_df["close"].to_list(),
#     )
# ]

# fig = make_subplots(
#     rows=5,
#     cols=1,
#     shared_xaxes=True,
#     vertical_spacing=0.025,
#     row_heights=[0.50, 0.15, 0.12, 0.12, 0.11],
#     subplot_titles=(
#         "Precio, VWAP, Bollinger y Donchian",
#         "Volumen",
#         "ATR 14",
#         "RSI 14",
#         "Momentum 10",
#     ),
# )

# fig.add_trace(
#     go.Candlestick(
#         x=plot_df["plot_x"].to_list(),
#         open=plot_df["open"].to_list(),
#         high=plot_df["high"].to_list(),
#         low=plot_df["low"].to_list(),
#         close=plot_df["close"].to_list(),
#         name="Precio",
#     ),
#     row=1,
#     col=1,
# )

# # Precio
# add_segmented_line(fig, plot_df, "vwap_daily", "VWAP", 1, "orange", width=2)

# # Bandas de Bollinger
# add_segmented_line(fig, plot_df, "bb_upper_20", "BB superior", 1, "#7f7f7f", "dot")
# add_segmented_line(fig, plot_df, "bb_middle_20", "BB media", 1, "#7f7f7f")
# add_segmented_line(fig, plot_df, "bb_lower_20", "BB inferior", 1, "#7f7f7f", "dot")

# # Canales de Donchian
# add_segmented_line(
#     fig, plot_df, "donchian_upper_20", "Donchian superior", 1, "#9467bd", "dash"
# )
# add_segmented_line(
#     fig, plot_df, "donchian_middle_20", "Donchian medio", 1, "#9467bd", "dot"
# )
# add_segmented_line(
#     fig, plot_df, "donchian_lower_20", "Donchian inferior", 1, "#9467bd", "dash"
# )

# # Volumen: vela alcista rojo, vela bajista verde
# fig.add_trace(
#     go.Bar(
#         x=plot_df["plot_x"].to_list(),
#         y=plot_df["volume"].to_list(),
#         marker_color=volume_colors,
#         name="Volumen",
#     ),
#     row=2,
#     col=1,
# )

# # Indicadores inferiores
# add_segmented_line(fig, plot_df, "atr_14", "ATR 14", 3, "#1f77b4", width=2)
# add_segmented_line(fig, plot_df, "rsi_14", "RSI 14", 4, "#ff7f0e", width=2)
# add_segmented_line(fig, plot_df, "momentum_10", "Momentum 10", 5, "#2ca02c", width=2)

# fig.add_hline(y=70, row=4, col=1, line_dash="dot", line_color="red")
# fig.add_hline(y=30, row=4, col=1, line_dash="dot", line_color="green")
# fig.update_yaxes(range=[0, 100], row=4, col=1)

# # Una marca de eje por inicio de cada día y ticker.
# day_starts = (
#     plot_df
#     .group_by(GROUPS, maintain_order=True)
#     .agg(pl.col("plot_x").min().alias("plot_x"))
#     .sort("plot_x")
# )

# fig.update_xaxes(
#     tickmode="array",
#     tickvals=day_starts["plot_x"].to_list(),
#     ticktext=[
#         f"{row['ticker']}<br>{row['trading_day']}"
#         for row in day_starts.to_dicts()
#     ],
#     row=5,
#     col=1,
# )

# # Etiqueta centrada sobre cada tramo de ticker.
# ticker_ranges = (
#     plot_df
#     .group_by("ticker", maintain_order=True)
#     .agg(
#         pl.col("plot_x").min().alias("start"),
#         pl.col("plot_x").max().alias("end"),
#     )
# )

# for ticker_range in ticker_ranges.to_dicts():
#     fig.add_annotation(
#         x=(ticker_range["start"] + ticker_range["end"]) / 2,
#         y=1.03,
#         xref="x",
#         yref="paper",
#         text=f"<b>{ticker_range['ticker']}</b>",
#         showarrow=False,
#     )

# fig.update_layout(
#     title="Precio e indicadores técnicos",
#     xaxis_rangeslider_visible=False,
#     height=1_300,
#     barmode="overlay",
# )

# fig.show()

Conversion de data para backtesting

In [ ]:
def get_data_to_backtesting(df,ticker,date):
    data = df_to_plot.filter(pl.col('ticker')==ticker).filter(pl.col('datetime').dt.date()==date)
    data.columns = [c[0].upper()+c[1:].lower() for c in df_to_plot.columns]
    data = data.to_pandas().set_index('Datetime')
    return data    



In [ ]:
from backtesting import Backtest, Strategy
import numpy as np


class OrbBreakout(Strategy):
    valor_R = 2

    def ORB_indicator(self):
        return np.vstack(
            (
                self.data.Orb_high_15m,
                self.data.Orb_low_15m,
            )
        )

    def vwap_indicator(self):
        return self.data.Vwap_daily

    def init(self):
        self.runner_activation_threshold = None
        self.runner_highest_high = None
        self.orb_high, self.orb_low = self.I(self.ORB_indicator)
        self.vwap = self.I(self.vwap_indicator)

    def next(self):
        runner_trade = next(
            (trade for trade in self.trades if trade.tag == "runner"),
            None,
        )
        if runner_trade is not None and self.runner_activation_threshold is not None:
            if self.runner_highest_high is None:
                if self.data.High[-1] >= self.runner_activation_threshold:
                    self.runner_highest_high = self.data.High[-1]
            else:
                self.runner_highest_high = max(
                    self.runner_highest_high,
                    self.data.High[-1],
                )

            if self.runner_highest_high is not None:
                trailing_stop = self.runner_highest_high - 2 * self.data.Atr_14[-1]
                if runner_trade.sl is None or trailing_stop > runner_trade.sl:
                    runner_trade.sl = trailing_stop

        orb_high = self.data.Orb_high_15m[-1]
        previous_orb_high = self.data.Orb_high_15m[-2]
        crossed_orb_high = (
            np.isfinite(orb_high)
            and np.isfinite(previous_orb_high)
            and self.data.Close[-1] > orb_high
            and self.data.Close[-2] <= previous_orb_high
        )

        recent_volumes = self.data.Volume[-4:-1]
        volume_surge = (
            len(recent_volumes) == 3
            and self.data.Volume[-1] > self.data.Median_vol_100[-1]
        )
        vwap_filter = self.data.Close < self.vwap * 1.05
        if crossed_orb_high and volume_surge and vwap_filter and not self.trades and not self.orders:
            atr = self.data.Atr_14[-1]
            entry_price = self.data.Close[-1]
            stop_loss = entry_price - atr * 2
            take_profit_1 = entry_price + atr * 2
            take_profit_2 = entry_price + atr * 4

            if np.isfinite(atr) and stop_loss > 0 and take_profit_1 > 0:
                size = int(self.valor_R / (entry_price - stop_loss))
                size_2 = int(size / 4)
                runner_size = int(size / 4)
                size_1 = size - size_2 - runner_size

                if size_1 > 0:
                    self.runner_activation_threshold = take_profit_2
                    self.runner_highest_high = None
                    self.buy(sl=stop_loss, tp=take_profit_1, size=size_1, tag="tp1")
                    self.buy(sl=stop_loss, tp=take_profit_2, size=size_2, tag="tp2")
                    self.buy(sl=stop_loss, size=runner_size, tag="runner")


In [ ]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
import numpy as np
from backtesting.test import SMA, GOOG


# Experiment: keep TP1 and the upper-wick exit for salida 2, while replacing the bearish-engulfing exit for salida 3 with a confirmed higher-low break.
class VwapCrossStructure(Strategy):


    max_bars = 5
    valor_R=2
    def bollinger_bands_indicator(self):
        return np.vstack(
            (
                self.data.Bb_upper_20,
                self.data.Bb_middle_20,
                self.data.Bb_lower_20,
            )
        )
    def donchian_channels_indicator(self):
            return np.vstack(
                (
                    self.data.Donchian_upper_20	,
                    self.data.Donchian_lower_20,
                    self.data.Donchian_middle_20,
                )
            )
    
    def vwap_indicator(self):
        return self.data.Vwap_daily
    def atr_indicator(self):
            return self.data.Atr_14
    def momentum_indicator(self):
            return self.data.Momentum_10
    def rsi_indicator(self):
         return self.data.Rsi_14
    def init(self):
        price = self.data.Close
        self.vwap = self.I(self.vwap_indicator)
        # self.atr = self.I(self.atr_indicator)
        self.momentum= self.I(self.momentum_indicator)
        self.rsi = self.I(self.rsi_indicator)

        # self.bb_upper, self.bb_middle, self.bb_lower = self.I(
        #     self.bollinger_bands_indicator,
        #     name=("BB superior", "BB media", "BB inferior"),
        #     overlay=True,
        # )
        # self.donchian_upper, self.donchian_middle, self.donchian_lower = self.I(
        #     self.donchian_channels_indicator,
        #     name=("donchian superior", "donchian media", "donchian inferior"),
        #     overlay=True,
        # )
        self.nbars=0
        # self.ma2 = self.I(SMA, price, 20)
        self.data_ant=None
        self.previous_pivot_low = None
        self.higher_low = None

    def next(self):
        # if crossover(self.data.Close, self.data.Vwap_daily) and not self.position.is_long:
        if self.data.Close[-1]> self.data.Vwap_daily[-1] and self.data.Close[-2]< self.data.Vwap_daily[-2] and not self.trades and not self.orders:
            recent_volumes = self.data.Volume[-4:-1]
            if len(recent_volumes) == 3 and (
                self.data.Volume[-1] * 30 >= sum(recent_volumes) * 11
            ):
                # sl= self.data.Low[-3:].min()
                # tp =self.data.Close+ (self.data.Close-sl)*2
                sl= self.data.Close[-1]-self.data.Atr_14[-1]*2
                sl2= self.data.Close[-1]-self.data.Atr_14[-1]*2
                sl3= self.data.Close[-1]-self.data.Atr_14[-1]*2
                tp =self.data.Close[-1]+self.data.Atr_14[-1]*2
                tp2 =self.data.Close[-1]+self.data.Atr_14[-1]*3
                tp3 =self.data.Close[-1]+self.data.Atr_14[-1]*6
                self.nbars=0
                self.previous_pivot_low = None
                self.higher_low = None
                if sl>0 and tp>0:
                    # print(self.data.Close[-1],sl,tp)

                    size = int(self.valor_R/(self.data.Close[-1]-sl))
                    size2=int (size/4)
                    size3=int (size/4)
                    size1 = size-size2-size3
                    # print(size)
                    # print(self.data.Close[-1],sl,tp,size,size1,size2,size3)
                    # self.buy(sl=sl,tp=tp,size=size1)
                    # self.buy(sl=sl2,tp=tp2,size=size2)
                    # self.buy(sl=sl3,tp=tp3,size=size3)
                    self.buy(sl=sl, tp=tp, size=size1, tag="salida 1")
                    self.buy(sl=sl2, size=size2, tag="salida 2")
                    self.buy(sl=sl3, size=size3, tag="salida 3")

        open_price = self.data.Open[-1]
        high_price = self.data.High[-1]
        low_price = self.data.Low[-1]
        close_price = self.data.Close[-1]
        previous_open = self.data.Open[-2]
        previous_close = self.data.Close[-2]
        
        body = abs(close_price - open_price)
        upper_wick = high_price - max(open_price, close_price)
        lower_wick = min(open_price, close_price) - low_price
        upper_wick_signal = (
            body > 0
            and upper_wick > body * 3
            and lower_wick < upper_wick*1.5
        )

        pivot_low_confirmed = (
            self.data.Low[-2] < self.data.Low[-3]
            and self.data.Low[-2] <= self.data.Low[-1]
        )

        salida_1_abierta = any(
            trade.tag == "salida 1" for trade in self.trades
        )

        if not salida_1_abierta:
            if pivot_low_confirmed:
                pivot_low = self.data.Low[-2]
                if (
                    self.previous_pivot_low is not None
                    and pivot_low > self.previous_pivot_low
                ):
                    self.higher_low = pivot_low
                self.previous_pivot_low = pivot_low

            for trade in self.trades:
                if trade.tag == "salida 2" and upper_wick_signal:
                    trade.close()

                if (
                    trade.tag == "salida 3"
                    and self.higher_low is not None
                    and close_price < self.higher_low
                ):
                    trade.close()
                    return
                # elif crossover(self.data.Close, self.data.vwap_daily):
                    # self.sell()
                # if self.position.is_long:
                #     self.nbars+=1
                #     if self.nbars>=self.max_bars:
                        
                #         self.position.close()
                self.data_ant = self.data


In [ ]:
from pathlib import Path
from datetime import timedelta
import math


def strategy_name(strategy):
    """Return the class name for a Backtesting.py strategy class or instance."""
    return strategy.__name__ if isinstance(strategy, type) else type(strategy).__name__


def build_backtest_output_dir(config, strategy, ticker, trading_date):
    """Build the output directory without creating it."""
    day = trading_date.date() if isinstance(trading_date, datetime) else trading_date
    return (
        Path(config["data_dir"])
        / "output"
        / strategy_name(strategy)
        / str(ticker)
        / day.strftime("%Y%m%d")
    )


def backtesting_stats_to_scalar_metrics(stats):
    """Keep only scalar Backtesting.py statistics suitable for Polars/Parquet."""
    metrics = {}
    for key, value in stats.items():
        if str(key).startswith("_"):
            continue

        if isinstance(value, np.generic):
            value = value.item()

        if isinstance(value, (datetime, date)):
            value = value.isoformat()
        elif isinstance(value, timedelta):
            value = value.total_seconds()
        elif isinstance(value, float) and not math.isfinite(value):
            value = None
        elif not isinstance(value, (str, int, float, bool, type(None))):
            continue

        metrics[str(key)] = value
    return metrics

In [ ]:
def run_and_persist_backtest(ticker, trading_date, strategy=VwapCross):
    """Run and persist one ticker/date backtest using the selected strategy settings."""
    day = trading_date.date() if isinstance(trading_date, datetime) else trading_date
    data = get_data_to_backtesting(df_to_plot, ticker, day)

    if data.empty:
        raise ValueError(f"No backtesting data available for {ticker} on {day:%Y-%m-%d}.")

    bt = Backtest(
        data,
        strategy,
        commission=(0.001, 0),
        cash=500,
        exclusive_orders=False,
    )
    stats = bt.run(valor_R=2)

    output_dir = build_backtest_output_dir(config, strategy, ticker, day)
    output_dir.mkdir(parents=True, exist_ok=True)

    result_row = {
        **backtesting_stats_to_scalar_metrics(stats),
        "strategy": strategy_name(strategy),
        "ticker": str(ticker),
        "date": day.isoformat(),
    }
    result = pl.DataFrame([result_row])
    result.write_parquet(output_dir / "stats.parquet")
    bt.plot(filename=str(output_dir / "backtest.html"), open_browser=False)
    return result


In [ ]:
def run_and_persist_backtest_batch(df_runs, strategy=VwapCross):
    """Run, persist, and aggregate every ticker/date combination in df_runs."""
    results = []
    current_strategy = strategy_name(strategy)

    for run in df_runs.select("ticker", "date").iter_rows(named=True):
        ticker = str(run["ticker"])
        trading_date = run["date"]
        day = trading_date.date() if isinstance(trading_date, datetime) else trading_date

        try:
            result = run_and_persist_backtest(ticker, trading_date, strategy=strategy).with_columns(
                pl.lit("success").alias("status"),
                pl.lit(None, dtype=pl.Utf8).alias("error_message"),
            )
        except Exception as exc:
            result = pl.DataFrame(
                [
                    {
                        "strategy": current_strategy,
                        "ticker": ticker,
                        "date": day.isoformat(),
                        "status": "failed",
                        "error_message": str(exc),
                    }
                ]
            )

        results.append(result)

    if results:
        aggregated = pl.concat(results, how="diagonal_relaxed").sort(["ticker", "date"])
    else:
        aggregated = pl.DataFrame(
            schema={
                "strategy": pl.Utf8,
                "ticker": pl.Utf8,
                "date": pl.Utf8,
                "status": pl.Utf8,
                "error_message": pl.Utf8,
            }
        )

    output_dir = Path(config["data_dir"]) / "output" / current_strategy
    output_dir.mkdir(parents=True, exist_ok=True)
    aggregated.write_parquet(output_dir / "results.parquet")
    return aggregated


In [ ]:
ticker = "VEEA"
trading_date = datetime.strptime("2026-10-09", "%Y-%m-%d")
backtest_result = run_and_persist_backtest(ticker, trading_date, strategy=OrbBreakout)
backtest_result


In [ ]:
df_backtest_results = run_and_persist_backtest_batch(df_backtest_runs, strategy=OrbBreakout)
df_backtest_results

In [ ]:
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots

kpi_columns = [
    "Return [%]",
    "Buy & Hold Return [%]",
    "Win Rate [%]",
    "Avg. Trade [%]",
    "Avg. Drawdown [%]",
    "Expectancy [%]",
]
strategy_colors = {
    "OrbBreakout": "#1f77b4",
}

results_dir = Path(config["data_dir"]) / "output"
result_paths = {
    strategy: results_dir / strategy / "results.parquet"
    for strategy in strategy_colors
}
missing_files = [str(path) for path in result_paths.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Cannot plot KPI comparisons because these result files are missing: "
        + ", ".join(missing_files)
    )

successful_results = {}
required_columns = ["status", *kpi_columns]
for strategy, path in result_paths.items():
    results = pl.read_parquet(path)
    missing_columns = [
        column for column in required_columns if column not in results.columns
    ]
    if missing_columns:
        raise ValueError(
            f"Cannot plot {strategy} KPI comparisons because its results are missing: "
            + ", ".join(missing_columns)
        )

    successful_results[strategy] = results.filter(pl.col("status") == "success")
    if successful_results[strategy].height == 0:
        raise ValueError(
            f"Cannot plot {strategy} KPI comparisons because there are no successful results."
        )

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=kpi_columns,
)

for index, kpi in enumerate(kpi_columns):
    row = index // 3 + 1
    col = index % 3 + 1

    for strategy, color in strategy_colors.items():
        values = successful_results[strategy].get_column(kpi).drop_nulls().to_list()
        fig.add_trace(
            go.Histogram(
                x=values,
                name=strategy,
                legendgroup=strategy,
                showlegend=index == 0,
                marker_color=color,
                opacity=0.55,
            ),
            row=row,
            col=col,
        )

    fig.update_xaxes(title_text=kpi, row=row, col=col)
    fig.update_yaxes(title_text="Count", row=row, col=col)

fig.update_layout(
    title="KPI distributions: OrbBreakout",
    barmode="overlay",
    height=700,
    width=1_300,
)
fig.show()